In [1]:
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, GPT2Model

In [2]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "flaitenberger/wnut_17"

BATCH_SIZE = 4
NUM_EPOCHS = 5

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", DEVICE)

torch.manual_seed(SEED)



Device: cuda


In [3]:
# ============================================================
# Load WNUT 2017
# ============================================================

dataset = load_dataset(DATASET_NAME)

print(dataset)

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))



README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  307kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.4kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3394 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1287 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})
Train size: 3394
Validation size: 1009
Test size: 1287


In [4]:
# ============================================================
# Inspect labels
# ============================================================

label_names = dataset["train"].features["ner_tags"].feature.names

NUM_LABELS = len(label_names)

print()
print("Number of labels:", NUM_LABELS)

for i, name in enumerate(label_names):
    print(i, name)




Number of labels: 13
0 O
1 B-corporation
2 I-corporation
3 B-creative-work
4 I-creative-work
5 B-group
6 I-group
7 B-location
8 I-location
9 B-person
10 I-person
11 B-product
12 I-product


In [5]:

# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

# GPT-2 doesn't have a native padding token.
tokenizer.pad_token = tokenizer.eos_token



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
# ============================================================
# Tokenization + NER label alignment
# ============================================================

def tokenize_and_align_labels(example):

    # The WNUT example contains something conceptually like:
    #
    # tokens:
    # ["Hello", "from", "New", "York"]
    #
    # ner_tags:
    # [0, 0, 7, 8]

    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        add_special_tokens=True,
    )

    word_ids = tokenized.word_ids()

    aligned_labels = []

    previous_word_id = None


    for word_id in word_ids:

        # Special tokens such as padding:
        #
        # word_id = None
        #
        # They should not contribute to the loss.

        if word_id is None:

            aligned_labels.append(-100)

        # First BPE subtoken of a word:
        elif word_id != previous_word_id:

            aligned_labels.append(
                example["ner_tags"][word_id]
            )

        # Additional BPE subtokens:
        else:

            # Ignore them for this first experiment.
            aligned_labels.append(-100)

        previous_word_id = word_id

    tokenized["labels"] = aligned_labels

    return tokenized


train_dataset = dataset["train"].map(
    tokenize_and_align_labels,
    remove_columns=dataset["train"].column_names,
)

val_dataset = dataset["validation"].map(
    tokenize_and_align_labels,
    remove_columns=dataset["validation"].column_names,
)



Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

In [7]:
an_example = dataset["train"][0]
print(an_example)
an_example_tokenized = tokenizer(
        an_example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        add_special_tokens=True,
    )
print(an_example_tokenized)
an_example_tokenized_word_ids = an_example_tokenized.word_ids()
print(an_example_tokenized_word_ids)
aligned_ner_tags = tokenize_and_align_labels(an_example)
print(aligned_ner_tags)

{'id': '0', 'tokens': ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two', 'weeks', '.', 'Empire', 'State', 'Building', '=', 'ESB', '.', 'Pretty', 'bad', 'storm', 'here', 'last', 'evening', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0]}
{'input_ids': [31, 79, 2518, 11152, 1026, 338, 1169, 1177, 6738, 3003, 40, 1101, 19950, 1640, 11545, 732, 2573, 13, 36, 35386, 9012, 25954, 28, 1546, 33, 13, 35700, 14774, 12135, 1456, 12957, 10197, 278, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 12, 13, 14, 14, 15, 16, 17, 18, 18, 19, 20, 21, 22, 23, 24, 25, 25, 26]
{'input_ids': [31, 79, 2518, 11152, 1026, 338, 1169, 1177, 6738, 3003, 40, 1101, 19950, 1640, 11545, 732, 2573, 13, 36, 35386, 9012, 25954, 28, 1546, 33, 13, 35700, 14774, 12135, 1456, 12957, 10197, 278, 13], 'atte

In [8]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(batch):

    # input_ids and attention_mask are dynamically padded.
    #
    # We cannot simply use tokenizer.pad() for labels because
    # labels need -100 padding rather than the pad token ID.

    input_ids = [
        item["input_ids"]
        for item in batch
    ]

    attention_masks = [
        item["attention_mask"]
        for item in batch
    ]

    labels = [
        item["labels"]
        for item in batch
    ]

    max_length = max(
        len(ids)
        for ids in input_ids
    )


    padded_input_ids = []
    padded_attention_masks = []
    padded_labels = []

    for ids, mask, label in zip(
        input_ids,
        attention_masks,
        labels,
    ):

        padding_length = (
            max_length - len(ids)
        )

        padded_input_ids.append(
            ids + [
                tokenizer.pad_token_id
            ] * padding_length
        )

        padded_attention_masks.append(
            mask + [0] * padding_length
        )

        padded_labels.append(
            label + [-100] * padding_length
        )


    return {
        "input_ids": torch.tensor(
            padded_input_ids,
            dtype=torch.long,
        ),

        "attention_mask": torch.tensor(
            padded_attention_masks,
            dtype=torch.long,
        ),

        "labels": torch.tensor(
            padded_labels,
            dtype=torch.long,
        ),
    }


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)


In [9]:
# ============================================================
# Model
# ============================================================

class GPT2NERClassifier(nn.Module):

    def __init__(
        self,
        model_name,
        num_labels,
    ):
        super().__init__()

        # GPT-2 backbone WITHOUT the LM head.
        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        hidden_size = self.gpt2.config.hidden_size

        # Token classification head:
        #
        # [B, T, 768]
        #       ↓
        # Linear(768 → num_labels)
        #       ↓
        # [B, T, num_labels]
        self.classifier = nn.Linear(
            hidden_size,
            num_labels,
        )


    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,
    ):

        # ----------------------------------------------------
        # GPT-2
        #
        # [B, T] → [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Token classification
        #
        # [B, T, 768]
        #       ↓
        # [B, T, K]
        # ----------------------------------------------------


        logits = self.classifier(
            hidden_states
        )

        loss = None

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss(
                ignore_index=-100
            )


            # CrossEntropyLoss expects:
            #
            # input:
            #     [N, K]
            #
            # target:
            #     [N]
            #
            # So flatten the B and T dimensions.
            #
            # logits:
            #     [B, T, K]
            #
            # → [B*T, K]
            #
            # labels:
            #     [B, T]
            #
            # → [B*T]

            loss = loss_fn(
                logits.view(-1, NUM_LABELS),
                labels.view(-1),
            )

        return {
            "loss": loss,
            "logits": logits,
        }



model = GPT2NERClassifier(
    MODEL_NAME,
    NUM_LABELS,
)



model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [10]:
# ============================================================
# Freeze GPT-2
# ============================================================

for parameter in model.gpt2.parameters():
    parameter.requires_grad = False




In [11]:
# ============================================================
# Parameter counts
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print()
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)




Total parameters: 124449805
Trainable parameters: 9997


In [12]:

# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    [
        p
        for p in model.parameters()
        if p.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)



In [13]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate1(model, data_loader):

    model.eval()

    total_loss = 0.0

    total_correct = 0
    total_valid_tokens = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]
        logits = outputs["logits"]


        predictions = logits.argmax(
            dim=-1
        )

        # Only evaluate positions that have
        # an actual NER label.
        valid_positions = (
            labels != -100
        )

        total_correct += (
            (
                predictions[valid_positions]
                == labels[valid_positions]
            )
            .sum()
            .item()
        )

        total_valid_tokens += (
            valid_positions
            .sum()
            .item()
        )

        total_loss += (
            loss.item()
            * total_valid_tokens
        )


    accuracy = (
        total_correct
        / total_valid_tokens
    )

    # For simplicity, calculate average loss
    # directly over batches here.
    #
    # Accuracy is the more intuitive metric for
    # understanding this first experiment.
    average_loss = (
        total_loss
        / len(data_loader)
        if len(data_loader) > 0
        else 0.0
    )

    return average_loss, accuracy



In [14]:
@torch.no_grad()
def evaluate(model, data_loader):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    total_correct = 0
    total_valid_tokens = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]
        logits = outputs["logits"]

        predictions = logits.argmax(dim=-1)

        valid_positions = labels != -100

        total_correct += (
            predictions[valid_positions]
            == labels[valid_positions]
        ).sum().item()

        total_valid_tokens += (
            valid_positions.sum().item()
        )

        total_loss += loss.item()
        total_batches += 1

    average_loss = (
        total_loss / total_batches
    )

    accuracy = (
        total_correct / total_valid_tokens
    )

    return average_loss, accuracy

In [15]:
# ============================================================
# Training
# ============================================================

model.to(DEVICE)

for epoch in range(NUM_EPOCHS):

    model.train()

    running_loss = 0.0

    for step, batch in enumerate(
        train_loader
    ):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % 100 == 0:

            average_loss = (
                running_loss / 100
            )

            print(
                f"Epoch {epoch + 1} "
                f"Step {step + 1} "
                f"Loss {average_loss:.4f}"
            )

            running_loss = 0.0

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
    )


    print()
    print(
        f"Epoch {epoch + 1} "
        f"Validation Loss: {val_loss:.4f} "
        f"Validation Accuracy: "
        f"{val_accuracy:.4f}"
    )
    print()




Epoch 1 Step 100 Loss 0.5287
Epoch 1 Step 200 Loss 0.3054
Epoch 1 Step 300 Loss 0.3029
Epoch 1 Step 400 Loss 0.3075
Epoch 1 Step 500 Loss 0.2155
Epoch 1 Step 600 Loss 0.2501
Epoch 1 Step 700 Loss 0.2256
Epoch 1 Step 800 Loss 0.2352

Epoch 1 Validation Loss: 0.3727 Validation Accuracy: 0.9232

Epoch 2 Step 100 Loss 0.2322
Epoch 2 Step 200 Loss 0.2356
Epoch 2 Step 300 Loss 0.1951
Epoch 2 Step 400 Loss 0.1926
Epoch 2 Step 500 Loss 0.2143
Epoch 2 Step 600 Loss 0.2102
Epoch 2 Step 700 Loss 0.1977
Epoch 2 Step 800 Loss 0.2232

Epoch 2 Validation Loss: 0.3536 Validation Accuracy: 0.9246

Epoch 3 Step 100 Loss 0.1660
Epoch 3 Step 200 Loss 0.2040
Epoch 3 Step 300 Loss 0.1969
Epoch 3 Step 400 Loss 0.1760
Epoch 3 Step 500 Loss 0.2128
Epoch 3 Step 600 Loss 0.1949
Epoch 3 Step 700 Loss 0.2187
Epoch 3 Step 800 Loss 0.1834

Epoch 3 Validation Loss: 0.3833 Validation Accuracy: 0.9237

Epoch 4 Step 100 Loss 0.1797
Epoch 4 Step 200 Loss 0.1720
Epoch 4 Step 300 Loss 0.1541
Epoch 4 Step 400 Loss 0.1692
Ep

In [16]:
# ============================================================
# Example NER prediction
# ============================================================

def predict_entities(model, words):

    model.eval()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    logits = outputs["logits"]

    predictions = logits.argmax(
        dim=-1
    )[0]


    word_ids = encoded.word_ids(
        batch_index=0
    )

    results = []

    previous_word_id = None

    for token_index, word_id in enumerate(
        word_ids
    ):

        if word_id is None:
            continue

        # Only use the first subtoken of
        # each word.
        if word_id == previous_word_id:
            continue

        predicted_label_id = predictions[
            token_index
        ].item()

        results.append(
            (
                words[word_id],
                label_names[predicted_label_id],
            )
        )


        previous_word_id = word_id

    return results


examples = [
    ["I", "live", "in", "San", "Francisco", "."],
    ["Apple", "released", "a", "new", "iPhone", "."],
    ["Michael", "visited", "London", "last", "week", "."],
]

for words in examples:

    results = predict_entities(
        model,
        words,
    )

    print()
    print("Input:", " ".join(words))

    for word, label in results:
        print(
            f"{word:15s} {label}"
        )




Input: I live in San Francisco .
I               O
live            O
in              O
San             B-location
Francisco       I-location
.               O

Input: Apple released a new iPhone .
Apple           O
released        O
a               O
new             O
iPhone          B-product
.               O

Input: Michael visited London last week .
Michael         O
visited         O
London          B-location
last            O
week            O
.               O
